In [1]:
#Importing Libraries
import pandas as pd
from datetime import datetime
import matplotlib.pyplot as plt
from yahoofinancials import YahooFinancials

In [2]:
ticker_details = pd.read_excel(r"C:\Users\Arun Sabarish\Documents\GitHub\Gold-Return-Prediction\Ticker List.xlsx")
ticker = ticker_details['Ticker'].to_list()
names = ticker_details['Description'].to_list()

In [3]:
#Preparing Date Range
end_date= datetime.strftime(datetime.today(),'%Y-%m-%d')
start_date = "2019-01-01"
date_range = pd.bdate_range(start=start_date,end=end_date)
values = pd.DataFrame({ 'Date': date_range})
values['Date']= pd.to_datetime(values['Date'])

#Extracting Data from Yahoo Finance and Adding them to Values table using date as key
for i in ticker:
    raw_data = YahooFinancials(i)
    raw_data = raw_data.get_historical_price_data(start_date, end_date, "daily")
    df = pd.DataFrame(raw_data[i]['prices'])[['formatted_date','adjclose']]
    df.columns = ['Date1',i]
    df['Date1']= pd.to_datetime(df['Date1'])
    values = values.merge(df,how='left',left_on='Date',right_on='Date1')
    values = values.drop(labels='Date1',axis=1)

#Renaming columns to represent instrument names rather than their ticker codes for ease of readability
names.insert(0,'Date')
values.columns = names

#Front filling the NaN values in the data set
values = values.fillna(method="ffill",axis=0)
values = values.fillna(method="bfill",axis=0)

# Co-ercing numeric type to all columns except Date
cols=values.columns.drop('Date')
values[cols] = values[cols].apply(pd.to_numeric,errors='coerce').round(decimals=1)
imp = ['Gold','Silver', 'Crude Oil', 'S&P500','MSCI EM ETF']

# Calculating Short term -Historical Returns
change_days = [1,3,5,14,21]

data = pd.DataFrame(data=values['Date'])
for i in change_days:
    x= values[cols].pct_change(periods=i).add_suffix("-T-"+str(i))
    data=pd.concat(objs=(data,x),axis=1)
    x=[]

# Calculating Long term Historical Returns
change_days = [60,90,180,250]

for i in change_days:
    x= values[imp].pct_change(periods=i).add_suffix("-T-"+str(i))
    data=pd.concat(objs=(data,x),axis=1)
    x=[]

#Calculating Moving averages for Gold
moving_avg = pd.DataFrame(values['Date'],columns=['Date'])
moving_avg['Date']=pd.to_datetime(moving_avg['Date'],format='%Y-%b-%d')
moving_avg['Gold/15SMA'] = (values['Gold']/(values['Gold'].rolling(window=15).mean()))-1
moving_avg['Gold/30SMA'] = (values['Gold']/(values['Gold'].rolling(window=30).mean()))-1
moving_avg['Gold/60SMA'] = (values['Gold']/(values['Gold'].rolling(window=60).mean()))-1
moving_avg['Gold/90SMA'] = (values['Gold']/(values['Gold'].rolling(window=90).mean()))-1
moving_avg['Gold/180SMA'] = (values['Gold']/(values['Gold'].rolling(window=180).mean()))-1
moving_avg['Gold/90EMA'] = (values['Gold']/(values['Gold'].ewm(span=90,adjust=True,ignore_na=True).mean()))-1
moving_avg['Gold/180EMA'] = (values['Gold']/(values['Gold'].ewm(span=180,adjust=True,ignore_na=True).mean()))-1
moving_avg = moving_avg.dropna(axis=0)
#Merging Moving Average values to the feature space

data['Date']=pd.to_datetime(data['Date'],format='%Y-%b-%d')
data = pd.merge(left=data,right=moving_avg,how='left',on='Date')
data = data[data['Gold-T-250'].notna()]
prediction_data = data.copy()

In [4]:
from pycaret.regression import *
#Loading the stored model
regressor_22 = load_model("22Day Regressor");
#Making Predictions
predicted_return_22 = predict_model(regressor_22,data=prediction_data)
predicted_return_22=predicted_return_22[['Date','Label']]
predicted_return_22.columns = ['Date','Return_22']


#Adding return Predictions to Gold Values
predicted_values = values[['Date','Gold']]
predicted_values = predicted_values.tail(len(predicted_return_22))
predicted_values = pd.merge(left=predicted_values,right=predicted_return_22,on=['Date'],how='inner')
predicted_values['Gold-T+22']=(predicted_values['Gold']*(1+predicted_values['Return_22'])).round(decimals =1)
#Adding T+22 Date
from datetime import datetime, timedelta
predicted_values['Date-T+22'] = predicted_values['Date']+timedelta(days = 22)
predicted_values.tail(n=10)

,Date,Gold,Return_22,Gold-T+22,Date-T+22
225,2020-10-27,1908.8,0.0035,1915.5,2020-11-18
226,2020-10-28,1876.2,-0.0263,1826.9,2020-11-19
227,2020-10-29,1865.6,-0.0096,1847.7,2020-11-20
228,2020-10-30,1877.4,-0.0005,1876.5,2020-11-21
229,2020-11-02,1890.4,0.0123,1913.7,2020-11-24
230,2020-11-03,1908.5,0.0242,1954.7,2020-11-25
231,2020-11-04,1894.6,0.0020,1898.4,2020-11-26
232,2020-11-05,1945.3,0.0081,1961.1,2020-11-27
233,2020-11-06,1950.3,0.0196,1988.5,2020-11-28
234,2020-11-09,1950.3,0.0019,1954.0,2020-12-01


In [5]:
from pycaret.regression import *
#Loading the stored model
regressor_14 = load_model("14Day Regressor");
#Making Predictions
predicted_return_14 = predict_model(regressor_14,data=prediction_data)
predicted_return_14=predicted_return_14[['Date','Label']]
predicted_return_14.columns = ['Date','Return_14']


#Adding return Predictions to Gold Values
predicted_values = values[['Date','Gold']]
predicted_values = predicted_values.tail(len(predicted_return_14))
predicted_values = pd.merge(left=predicted_values,right=predicted_return_14,on=['Date'],how='inner')
predicted_values['Gold-T+14']=(predicted_values['Gold']*(1+predicted_values['Return_14'])).round(decimals =1)
#Adding T+14 Date
from datetime import datetime, timedelta
predicted_values['Date-T+14'] = predicted_values['Date']+timedelta(days = 14)
predicted_values.tail(n=10)

,Date,Gold,Return_14,Gold-T+14,Date-T+14
225,2020-10-27,1908.8,0.0107,1929.2,2020-11-10
226,2020-10-28,1876.2,-0.0115,1854.6,2020-11-11
227,2020-10-29,1865.6,0.0021,1869.5,2020-11-12
228,2020-10-30,1877.4,0.0077,1891.9,2020-11-13
229,2020-11-02,1890.4,0.0197,1927.6,2020-11-16
230,2020-11-03,1908.5,0.0223,1951.1,2020-11-17
231,2020-11-04,1894.6,-0.0070,1881.3,2020-11-18
232,2020-11-05,1945.3,0.0161,1976.6,2020-11-19
233,2020-11-06,1950.3,0.0096,1969.0,2020-11-20
234,2020-11-09,1950.3,0.0048,1959.7,2020-11-23
